In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [8]:

# Load data
df = pd.read_csv('exp1_high_entropy.csv')



In [ ]:
fig = plt.figure(figsize=(20, 10))

colors = {
    'DATL-TAIA': '#27AE60',
    'Oyamada-TinyLLM': '#3498DB',
    'LSTM': '#E67E22',
    'XGBoost': '#95A5A6',
    'MT-RNN': '#E74C3C'
}

datasets = df['Dataset'].unique()
models = df['Model'].unique()

# ========== NA Accuracy Bar Chart ==========
ax1 = plt.subplot(2, 3, 1)
x = np.arange(len(datasets))
width = 0.16

for i, model in enumerate(models):
    nas = [df[(df['Dataset']==d) & (df['Model']==model)]['NA_Acc'].values[0] for d in datasets]
    cis = [df[(df['Dataset']==d) & (df['Model']==model)]['NA_CI'].values[0] for d in datasets]

    offset = (i - len(models)/2) * width
    bars = ax1.bar(x + offset, nas, width, label=model, color=colors[model],
                  edgecolor='black', linewidth=2, alpha=0.9)
    ax1.errorbar(x + offset, nas, yerr=cis, fmt='none', ecolor='black',
                capsize=6, capthick=2, linewidth=1.5)

    if model == 'DATL-TAIA':
        for bar in bars:
            bar.set_linewidth(3)
            bar.set_edgecolor('darkgreen')

ax1.set_ylabel('NA Accuracy (%) ± CI', fontweight='bold', fontsize=12)
ax1.set_title('High Entropy: NA Accuracy', fontweight='bold', fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels([d.replace('BPI', '') for d in datasets], fontsize=10)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# ========== NA F1 Bar Chart ==========
ax2 = plt.subplot(2, 3, 2)

for i, model in enumerate(models):
    f1s = [df[(df['Dataset']==d) & (df['Model']==model)]['NA_F1'].values[0] for d in datasets]
    cis = [df[(df['Dataset']==d) & (df['Model']==model)]['F1_CI'].values[0] for d in datasets]

    offset = (i - len(models)/2) * width
    bars = ax2.bar(x + offset, f1s, width, color=colors[model],
                  edgecolor='black', linewidth=2, alpha=0.9)
    ax2.errorbar(x + offset, f1s, yerr=cis, fmt='none', ecolor='black',
                capsize=6, capthick=2, linewidth=1.5)

    if model == 'DATL-TAIA':
        for bar in bars:
            bar.set_linewidth(3)
            bar.set_edgecolor('darkgreen')

ax2.set_ylabel('NA F1-Macro ± CI', fontweight='bold', fontsize=12)
ax2.set_title('High Entropy: F1-Macro', fontweight='bold', fontsize=13)
ax2.set_xticks(x)
ax2.set_xticklabels([d.replace('BPI', '') for d in datasets], fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# ========== RT MAE Bar Chart ==========
ax3 = plt.subplot(2, 3, 3)

for i, model in enumerate(models):
    maes = [df[(df['Dataset']==d) & (df['Model']==model)]['RT_MAE'].values[0] for d in datasets]
    cis = [df[(df['Dataset']==d) & (df['Model']==model)]['MAE_CI'].values[0] for d in datasets]

    offset = (i - len(models)/2) * width
    ax3.bar(x + offset, maes, width, color=colors[model],
           edgecolor='black', linewidth=2, alpha=0.9)
    ax3.errorbar(x + offset, maes, yerr=cis, fmt='none', ecolor='black',
                capsize=6, capthick=2, linewidth=1.5)

ax3.set_ylabel('RT MAE (days) ± CI', fontweight='bold', fontsize=12)
ax3.set_title('High Entropy: RT MAE', fontweight='bold', fontsize=13)
ax3.set_xticks(x)
ax3.set_xticklabels([d.replace('BPI', '') for d in datasets], fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')

# ========== Summary Table ==========
ax4 = plt.subplot(2, 3, (4, 6))
ax4.axis('off')

avg_stats = df.groupby('Model').agg({
    'NA_Acc': 'mean',
    'NA_CI': 'mean',
    'NA_F1': 'mean',
    'F1_CI': 'mean',
    'RT_MAE': 'mean',
    'MAE_CI': 'mean'
}).round(2).sort_values('NA_Acc', ascending=False).reset_index()

table_data = [['Model', 'Avg NA Acc ± CI', 'Avg F1 ± CI', 'Avg RT MAE ± CI']]

for _, row in avg_stats.iterrows():
    table_data.append([
        row['Model'],
        f"{row['NA_Acc']:.1f}% ± {row['NA_CI']:.1f}",
        f"{row['NA_F1']:.3f} ± {row['F1_CI']:.3f}",
        f"{row['RT_MAE']:.1f} ± {row['MAE_CI']:.1f} days"
    ])

table = ax4.table(cellText=table_data, cellLoc='center', loc='center',
                 colWidths=[0.25, 0.25, 0.25, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

for i in range(len(table_data)):
    for j in range(4):
        cell = table[(i, j)]
        if i == 0:
            cell.set_facecolor('#3498DB')
            cell.set_text_props(weight='bold', color='white', fontsize=12)
        elif 'DATL' in str(table_data[i][0]):
            cell.set_facecolor('#D5F4E6')
            cell.set_text_props(weight='bold')

ax4.set_title('Exp1: High Entropy Performance Summary\n(BPI2015 datasets)',
             fontweight='bold', fontsize=14, pad=20)

plt.suptitle('Experiment 1: High Entropy Datasets',
            fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('exp1_high_entropy_viz.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization complete")
print("\nAverage Performance:")
print(avg_stats[['Model', 'NA_Acc', 'NA_F1', 'RT_MAE']].to_string(index=False))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load data
df = pd.read_csv('exp1_high_entropy.csv')

fig = plt.figure(figsize=(20, 10))

colors = {
    'DATL-TAIA': '#27AE60',
    'Oyamada-TinyLLM': '#3498DB',
    'LSTM': '#E67E22',
    'XGBoost': '#95A5A6',
    'MT-RNN': '#E74C3C'
}

datasets = df['Dataset'].unique()
models = df['Model'].unique()

# ========== NA Accuracy Bar Chart ==========
ax1 = plt.subplot(2, 3, 1)
x = np.arange(len(datasets))
width = 0.16

for i, model in enumerate(models):
    nas = [df[(df['Dataset']==d) & (df['Model']==model)]['NA_Acc'].values[0] for d in datasets]
    cis = [df[(df['Dataset']==d) & (df['Model']==model)]['NA_CI'].values[0] for d in datasets]

    offset = (i - len(models)/2) * width
    bars = ax1.bar(x + offset, nas, width, label=model, color=colors[model],
                  edgecolor='black', linewidth=2, alpha=0.9)
    ax1.errorbar(x + offset, nas, yerr=cis, fmt='none', ecolor='black',
                capsize=6, capthick=2, linewidth=1.5)

    if model == 'DATL-TAIA':
        for bar in bars:
            bar.set_linewidth(3)
            bar.set_edgecolor('darkgreen')

ax1.set_ylabel('NA Accuracy (%) ± CI', fontweight='bold', fontsize=12)
ax1.set_title('High Entropy: NA Accuracy', fontweight='bold', fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels([d.replace('BPI', '') for d in datasets], fontsize=10)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# ========== NA F1 Bar Chart ==========
ax2 = plt.subplot(2, 3, 2)

for i, model in enumerate(models):
    f1s = [df[(df['Dataset']==d) & (df['Model']==model)]['NA_F1'].values[0] for d in datasets]
    cis = [df[(df['Dataset']==d) & (df['Model']==model)]['F1_CI'].values[0] for d in datasets]

    offset = (i - len(models)/2) * width
    bars = ax2.bar(x + offset, f1s, width, color=colors[model],
                  edgecolor='black', linewidth=2, alpha=0.9)
    ax2.errorbar(x + offset, f1s, yerr=cis, fmt='none', ecolor='black',
                capsize=6, capthick=2, linewidth=1.5)

    if model == 'DATL-TAIA':
        for bar in bars:
            bar.set_linewidth(3)
            bar.set_edgecolor('darkgreen')

ax2.set_ylabel('NA F1-Macro ± CI', fontweight='bold', fontsize=12)
ax2.set_title('High Entropy: F1-Macro', fontweight='bold', fontsize=13)
ax2.set_xticks(x)
ax2.set_xticklabels([d.replace('BPI', '') for d in datasets], fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# ========== RT MAE Bar Chart ==========
ax3 = plt.subplot(2, 3, 3)

for i, model in enumerate(models):
    maes = [df[(df['Dataset']==d) & (df['Model']==model)]['RT_MAE'].values[0] for d in datasets]
    cis = [df[(df['Dataset']==d) & (df['Model']==model)]['MAE_CI'].values[0] for d in datasets]

    offset = (i - len(models)/2) * width
    ax3.bar(x + offset, maes, width, color=colors[model],
           edgecolor='black', linewidth=2, alpha=0.9)
    ax3.errorbar(x + offset, maes, yerr=cis, fmt='none', ecolor='black',
                capsize=6, capthick=2, linewidth=1.5)

ax3.set_ylabel('RT MAE (days) ± CI', fontweight='bold', fontsize=12)
ax3.set_title('High Entropy: RT MAE', fontweight='bold', fontsize=13)
ax3.set_xticks(x)
ax3.set_xticklabels([d.replace('BPI', '') for d in datasets], fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')

# ========== Summary Table ==========
ax4 = plt.subplot(2, 3, (4, 6))
ax4.axis('off')

avg_stats = df.groupby('Model').agg({
    'NA_Acc': 'mean',
    'NA_CI': 'mean',
    'NA_F1': 'mean',
    'F1_CI': 'mean',
    'RT_MAE': 'mean',
    'MAE_CI': 'mean'
}).round(2).sort_values('NA_Acc', ascending=False).reset_index()

table_data = [['Model', 'Avg NA Acc ± CI', 'Avg F1 ± CI', 'Avg RT MAE ± CI']]

for _, row in avg_stats.iterrows():
    table_data.append([
        row['Model'],
        f"{row['NA_Acc']:.1f}% ± {row['NA_CI']:.1f}",
        f"{row['NA_F1']:.3f} ± {row['F1_CI']:.3f}",
        f"{row['RT_MAE']:.1f} ± {row['MAE_CI']:.1f} days"
    ])

table = ax4.table(cellText=table_data, cellLoc='center', loc='center',
                 colWidths=[0.25, 0.25, 0.25, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

for i in range(len(table_data)):
    for j in range(4):
        cell = table[(i, j)]
        if i == 0:
            cell.set_facecolor('#3498DB')
            cell.set_text_props(weight='bold', color='white', fontsize=12)
        elif 'DATL' in str(table_data[i][0]):
            cell.set_facecolor('#D5F4E6')
            cell.set_text_props(weight='bold')

ax4.set_title('Exp1: High Entropy Performance Summary\n(BPI2015 datasets)',
             fontweight='bold', fontsize=14, pad=20)

plt.suptitle('Experiment 1: High Entropy Datasets',
            fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('exp1_high_entropy_viz.png', dpi=300, bbox_inches='tight')
print("✓ Saved: exp1_high_entropy_viz.png")

plt.show()

print("\nAverage Performance:")
print(avg_stats[['Model', 'NA_Acc', 'NA_F1', 'RT_MAE']].to_string(index=False))